# 🧠 FaceMesh – Gesichtserkennung im Browser
### KI läuft komplett lokal – kein Server, keine Cloud

Dieses Notebook demonstriert **On-Device Machine Learning** mit [TensorFlow.js](https://www.tensorflow.org/js).  
Das KI-Modell **MediaPipe FaceMesh** erkennt ~478 Punkte im Gesicht in Echtzeit – direkt im Browser.

---
**Lernziele**
- Verstehen, wie KI-Modelle im Browser ausgeführt werden
- Unterschied zwischen Server-seitiger und Client-seitiger KI
- Canvas-API und Overlay-Zeichnung in JavaScript


## 1 · Hintergrund: Wie funktioniert FaceMesh?

FaceMesh ist ein neuronales Netz, das auf Millionen von Gesichtsbildern trainiert wurde.  
Als Eingabe bekommt es ein einzelnes Bild – als Ausgabe liefert es **478 (x, y, z)-Koordinaten**,  
die ein 3D-Gitter über das Gesicht legen.

```
Kamerabild  →  [FaceMesh-Modell]  →  478 Punkte (x, y, z)
                    ↑
          läuft im Browser (WebGL)
```

Die Koordinaten werden dann mit der **Canvas-API** als Overlay gezeichnet.


## 2 · Die App ausführen

Die nächste Zelle bettet die komplette App als interaktives HTML-Element ein.  
**Wichtig:** Der Browser fragt nach Kamera-Zugriff – bitte erlauben.


In [ ]:
from IPython.display import HTML

html_code = """
<!DOCTYPE html>
<html lang="de">
<head>
<meta charset="UTF-8"/>
<style>
  :root {
    --bg: #0a0a0f; --surface: #13131a; --border: #1e1e2e;
    --accent: #00ffe5; --accent2: #ff2d78;
    --text: #e8e8f0; --muted: #555568; --radius: 10px;
    --font: "Courier New", monospace;
  }
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body {
    background: var(--bg); color: var(--text);
    font-family: var(--font); padding: 16px;
    display: flex; flex-direction: column; align-items: center; gap: 12px;
  }
  h2 { font-size: .95rem; letter-spacing: .18em; color: var(--accent); text-transform: uppercase; }
  #status {
    font-size: .68rem; letter-spacing: .14em; padding: 4px 14px;
    border: 1px solid var(--border); border-radius: 999px; color: var(--muted); transition: all .3s;
  }
  #status.ready  { color: var(--accent);  border-color: var(--accent);  box-shadow: 0 0 10px #00ffe530; }
  #status.detect { color: var(--accent2); border-color: var(--accent2); animation: blink 1.2s infinite; }
  #status.error  { color: #ff4444; border-color: #ff4444; }
  @keyframes blink { 50% { opacity: .4; } }
  #wrap {
    position: relative; width: 480px; max-width: 100%; aspect-ratio: 4/3;
    background: var(--surface); border: 1px solid var(--border); border-radius: var(--radius); overflow: hidden;
  }
  #video  { display: none; }
  #canvas { position: absolute; inset: 0; width: 100%; height: 100%; }
  #placeholder {
    position: absolute; inset: 0; display: flex; align-items: center; justify-content: center;
    color: var(--muted); font-size: .75rem; letter-spacing: .1em;
  }
  button {
    padding: 10px 28px; background: transparent; border: 1px solid var(--accent);
    color: var(--accent); font-family: var(--font); font-size: .78rem;
    letter-spacing: .14em; text-transform: uppercase; border-radius: var(--radius);
    cursor: pointer; transition: background .2s, box-shadow .2s;
  }
  button:hover    { background: #00ffe510; box-shadow: 0 0 16px #00ffe530; }
  button:disabled { border-color: var(--muted); color: var(--muted); cursor: default; background: transparent; box-shadow: none; }
  #options { display: flex; gap: 8px; flex-wrap: wrap; justify-content: center; }
  .tog {
    display: flex; align-items: center; gap: 5px; font-size: .65rem; letter-spacing: .1em;
    color: var(--muted); cursor: pointer; padding: 5px 12px; border: 1px solid var(--border);
    border-radius: 999px; transition: all .2s; user-select: none;
  }
  .tog input { display: none; }
  .tog.on       { border-color: var(--accent); color: var(--accent); }
  .tog.on.pink  { border-color: var(--accent2); color: var(--accent2); }
  .dot { width: 7px; height: 7px; border-radius: 50%; background: var(--muted); transition: background .2s; }
  .tog.on .dot      { background: var(--accent); }
  .tog.on.pink .dot { background: var(--accent2); }
  #stats { display: flex; gap: 10px; }
  .card { background: var(--surface); border: 1px solid var(--border); border-radius: var(--radius); padding: 10px 16px; text-align: center; min-width: 90px; }
  .cl { font-size: .58rem; letter-spacing: .12em; color: var(--muted); margin-bottom: 3px; }
  .cv { font-size: 1.2rem; color: var(--accent); }
  #hint { font-size: .6rem; color: var(--muted); text-align: center; min-height: 1em; }
</style>
</head>
<body>
<h2>FaceMesh · TensorFlow.js</h2>
<div id="status">LADE MODELL …</div>
<div id="wrap">
  <video id="video" autoplay playsinline muted></video>
  <canvas id="canvas"></canvas>
  <div id="placeholder">▶ KAMERA STARTEN</div>
</div>
<button id="btn" disabled>LADE MODELL …</button>
<div id="options">
  <label class="tog on"  id="tMesh"  ><input type="checkbox" checked onchange="opt.mesh=this.checked;sync('tMesh',this)"   ><span class="dot"></span>MESH</label>
  <label class="tog on"  id="tCont"  ><input type="checkbox" checked onchange="opt.contour=this.checked;sync('tCont',this)"><span class="dot"></span>KONTUREN</label>
  <label class="tog"     id="tIrisU" ><input type="checkbox"         onchange="onIrisUngenau(this)"                        ><span class="dot"></span>IRIS UNGENAU</label>
  <label class="tog pink" id="tIrisP"><input type="checkbox"         onchange="onIrisPraezise(this)"                       ><span class="dot"></span>IRIS PRÄZISE</label>
</div>
<div id="hint">IRIS UNGENAU: schnell, aus Augenkontур · IRIS PRÄZISE: langsamer, echter Pupillenkreis</div>
<div id="stats">
  <div class="card"><div class="cl">GESICHTER</div><div class="cv" id="sFaces">–</div></div>
  <div class="card"><div class="cl">FPS</div><div class="cv" id="sFps">–</div></div>
</div>

<script src="https://cdn.jsdelivr.net/npm/@tensorflow/tfjs@4.17.0/dist/tf.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/@tensorflow-models/face-landmarks-detection@1.0.5/dist/face-landmarks-detection.js"></script>
<script>
const video   = document.getElementById('video');
const canvas  = document.getElementById('canvas');
const ctx     = canvas.getContext('2d');
const btn     = document.getElementById('btn');
const statusEl= document.getElementById('status');
const ph      = document.getElementById('placeholder');

const offscreen = document.createElement('canvas');
const offCtx    = offscreen.getContext('2d');

let detector=null, running=false, lastTs=0;
let refineMode = false;

// iris: 'none' | 'ungenau' | 'praezise'
let irisMode = 'none';

const opt = { mesh:true, contour:true };

const C_CONT = 'rgba(0,255,229,.85)';
const C_MESH = 'rgba(0,255,229,.20)';
const C_IRIS = 'rgba(255,45,120,.9)';

const OVAL=[10,338,297,332,284,251,389,356,454,323,361,288,397,365,379,378,400,377,152,148,176,149,150,136,172,58,132,93,234,127,162,21,54,103,67,109];
const LEYE=[362,382,381,380,374,373,390,249,263,466,388,387,386,385,384,398];
const REYE=[33,7,163,144,145,153,154,155,133,173,157,158,159,160,161,246];
const LEBR=[276,283,282,295,285,300,293,334,296,336];
const REBR=[46,53,52,65,55,70,63,105,66,107];
const LOUT=[61,185,40,39,37,0,267,269,270,409,291,375,321,405,314,17,84,181,91,146];
const LINN=[78,191,80,81,82,13,312,311,310,415,308,324,318,402,317,14,87,178,88,95];
const NOSE=[6,197,195,5,4,1,19,94,2];

// "Iris ungenau" an → "Iris präzise" aus, Modell auf refineLandmarks:false
function onIrisUngenau(cb) {
  if (cb.checked) {
    // Präzise ausschalten
    const pCb = document.querySelector('#tIrisP input');
    pCb.checked = false;
    setTog('tIrisP', false, true);
    irisMode = 'ungenau';
    setTog('tIrisU', true, false);
    if (refineMode) applyRefine(false);
  } else {
    irisMode = 'none';
    setTog('tIrisU', false, false);
  }
}

// "Iris präzise" an → "Iris ungenau" aus, Modell auf refineLandmarks:true
function onIrisPraezise(cb) {
  if (cb.checked) {
    // Ungenau ausschalten
    const uCb = document.querySelector('#tIrisU input');
    uCb.checked = false;
    setTog('tIrisU', false, false);
    irisMode = 'praezise';
    setTog('tIrisP', true, true);
    if (!refineMode) applyRefine(true);
  } else {
    irisMode = 'none';
    setTog('tIrisP', false, true);
  }
}

function setTog(id, on, pink) {
  const el = document.getElementById(id);
  el.classList.toggle('on', on);
  el.classList.toggle('pink', on && pink);
}

async function applyRefine(value) {
  if (refineMode === value) return;
  refineMode = value;
  const wasRunning = running;
  running = false;
  setStatus('LADE MODELL …','');
  btn.disabled = true;
  try {
    detector = await buildDetector(refineMode);
    setStatus(wasRunning ? 'ERKENNE …' : 'BEREIT', wasRunning ? 'detect' : 'ready');
    btn.disabled = false;
    if (wasRunning) { running = true; requestAnimationFrame(loop); }
  } catch(e) {
    setStatus('FEHLER: ' + e.message,'error');
  }
}

async function buildDetector(refine) {
  return await faceLandmarksDetection.createDetector(
    faceLandmarksDetection.SupportedModels.MediaPipeFaceMesh,
    { runtime:'tfjs', refineLandmarks: refine, maxFaces:2 }
  );
}

async function loadModel() {
  try {
    await tf.ready();
    detector = await buildDetector(false);
    setStatus('BEREIT','ready');
    btn.textContent='KAMERA STARTEN';
    btn.disabled=false;
  } catch(e) {
    setStatus('FEHLER: ' + e.message,'error');
  }
}

async function startCam() {
  const stream = await navigator.mediaDevices.getUserMedia({ video: true });
  video.srcObject = stream;
  await new Promise(r => video.onloadedmetadata = r);
  await video.play();
  await new Promise(r => setTimeout(r, 500));
  const w = video.videoWidth  || 640;
  const h = video.videoHeight || 480;
  canvas.width = offscreen.width  = w;
  canvas.height= offscreen.height = h;
  ph.style.display = 'none';
}

async function loop(ts) {
  if (!running) return;
  const fps = Math.round(1000 / ((ts - lastTs) || 1));
  lastTs = ts;
  document.getElementById('sFps').textContent = fps > 0 && fps < 999 ? fps : '–';

  offCtx.drawImage(video, 0, 0, offscreen.width, offscreen.height);
  const faces = await detector.estimateFaces(offscreen, { flipHorizontal: false });
  document.getElementById('sFaces').textContent = faces.length;

  ctx.save();
  ctx.translate(canvas.width, 0);
  ctx.scale(-1, 1);
  ctx.drawImage(video, 0, 0, canvas.width, canvas.height);
  faces.forEach(f => drawFace(f.keypoints));
  ctx.restore();

  requestAnimationFrame(loop);
}

function drawFace(kp) {
  if (opt.mesh) {
    ctx.fillStyle = C_MESH;
    kp.forEach(p => { ctx.beginPath(); ctx.arc(p.x,p.y,1.4,0,Math.PI*2); ctx.fill(); });
  }
  if (opt.contour) {
    [OVAL,LEYE,REYE,LEBR,REBR,LOUT,LINN,NOSE].forEach(idx => {
      ctx.beginPath(); ctx.strokeStyle=C_CONT; ctx.lineWidth=1.2;
      ctx.moveTo(kp[idx[0]].x, kp[idx[0]].y);
      idx.slice(1).forEach(i => ctx.lineTo(kp[i].x, kp[i].y));
      ctx.closePath(); ctx.stroke();
    });
  }
  if (irisMode === 'ungenau') {
    // Iris-Kreis aus Augenkontур schätzen (kein refineLandmarks nötig)
    [REYE, LEYE].forEach(idx => {
      const pts = idx.map(i => kp[i]).filter(Boolean);
      if (pts.length < 4) return;
      const cx = pts.reduce((s,p)=>s+p.x,0)/pts.length;
      const cy = pts.reduce((s,p)=>s+p.y,0)/pts.length;
      const r  = Math.max(...pts.map(p=>Math.hypot(p.x-cx,p.y-cy))) * 0.45;
      ctx.beginPath(); ctx.arc(cx,cy,r,0,Math.PI*2);
      ctx.strokeStyle=C_IRIS; ctx.lineWidth=2; ctx.stroke();
      ctx.beginPath(); ctx.arc(cx,cy,2,0,Math.PI*2);
      ctx.fillStyle=C_IRIS; ctx.fill();
    });
  }
  if (irisMode === 'praezise') {
    // Exakte Iris-Landmarks (nur mit refineLandmarks:true)
    ['rightIris','leftIris'].forEach(side => {
      const pts = kp.filter(p=>p.name&&p.name.startsWith(side));
      if (pts.length < 2) return;
      const cx = pts.reduce((s,p)=>s+p.x,0)/pts.length;
      const cy = pts.reduce((s,p)=>s+p.y,0)/pts.length;
      const t  = pts.find(p=>p.name.includes('Top'));
      const b  = pts.find(p=>p.name.includes('Bottom'));
      const r  = t&&b ? Math.abs(b.y-t.y)/2 : 8;
      ctx.beginPath(); ctx.arc(cx,cy,r,0,Math.PI*2);
      ctx.strokeStyle=C_IRIS; ctx.lineWidth=2; ctx.stroke();
      ctx.beginPath(); ctx.arc(cx,cy,2,0,Math.PI*2);
      ctx.fillStyle=C_IRIS; ctx.fill();
    });
  }
}

btn.addEventListener('click', async () => {
  if (running) {
    running = false;
    btn.textContent = 'KAMERA STARTEN';
    setStatus('BEREIT','ready');
    video.srcObject?.getTracks().forEach(t=>t.stop());
    video.srcObject = null;
    ctx.clearRect(0,0,canvas.width,canvas.height);
    ph.style.display = 'flex';
  } else {
    btn.disabled = true;
    setStatus('KAMERA STARTET …','');
    try {
      await startCam();
      running = true;
      btn.textContent = 'STOPPEN';
      btn.disabled = false;
      setStatus('ERKENNE …','detect');
      requestAnimationFrame(loop);
    } catch(e) {
      setStatus('KAMERA-FEHLER','error');
      btn.disabled = false;
    }
  }
});

function setStatus(t,c){ statusEl.textContent=t; statusEl.className=c; }
function sync(id,cb){ document.getElementById(id).classList.toggle('on',cb.checked); }

loadModel();
</script>
</body></html>
"""
HTML(f'<iframe srcdoc="{html_code.replace(chr(34), "&quot;").replace(chr(10), "&#10;")}" width="560" height="700" allow="camera" style="border:none;border-radius:12px;"></iframe>')

## 3 · Code-Erklärung: Die wichtigsten Konzepte

### 3.1 Modell laden
```python
# In JavaScript (läuft im Browser):
detector = await faceLandmarksDetection.createDetector(
    faceLandmarksDetection.SupportedModels.MediaPipeFaceMesh,
    { runtime: 'tfjs', refineLandmarks: true, maxFaces: 2 }
)
```
`runtime: 'tfjs'` bedeutet: Das Modell läuft komplett lokal via WebGL – **kein Server nötig**.

### 3.2 Inferenz pro Frame
```javascript
const faces = await detector.estimateFaces(video)
// → [ { keypoints: [ {x, y, z, name}, ... ] } ]
```
`estimateFaces()` ist der eigentliche KI-Aufruf. Er gibt pro Gesicht ein Array von ~478 Punkten zurück.

### 3.3 Overlay zeichnen
```javascript
ctx.arc(point.x, point.y, 1.5, 0, Math.PI * 2)
```
Die Koordinaten werden direkt auf ein `<canvas>` gezeichnet, das über dem Video-Element liegt.


## 4 · Aufgaben zum Experimentieren

**Aufgabe 1 – Farben ändern**  
Finde im Code die Zeilen mit `C_MESH`, `C_CONT`, `C_IRIS` und ändere die Farben.  
Tipp: `rgba(255, 200, 0, 0.8)` ergibt ein goldenes Gelb.

**Aufgabe 2 – Nur Augen anzeigen**  
Deaktiviere Mesh und Oval-Kontur. Aktiviere nur die Augen-Toggles.  
Was passiert, wenn du `maxFaces: 2` auf `maxFaces: 4` änderst?

**Aufgabe 3 – Augen-Blinzel-Erkennung** *(Fortgeschritten)*  
Der Y-Abstand zwischen Punkt `159` (Oberlid) und `145` (Unterlid) gibt die Augenöffnung an.  
Wenn dieser Abstand unter einen Schwellenwert fällt: Blinzeln erkannt!  
```javascript
const eyeOpen = Math.abs(kp[159].y - kp[145].y);
if (eyeOpen < 5) console.log("Blinzeln!");
```


## 5 · Weiterführende Fragen

- Warum spiegeln wir den Video-Feed (`transform: scaleX(-1)`)?
- Was bedeutet `requestAnimationFrame` – warum nicht `setInterval`?
- Wie viele Berechnungen führt das Modell pro Sekunde durch? (FPS × 478 Punkte)
- Was ist der Unterschied zwischen `runtime: 'tfjs'` und `runtime: 'mediapipe'`?
